# Lab 7 - Agentic Node Graph: the full pipeline

Doc 2.10 Lab 7, line 686: *"Wire the specialist agents into one explicit graph
with state and branching."* The output is *"An executable graph producing a first
end-to-end draft."* and the acceptance criterion, line 690, is *"One command
yields a validated organised course workspace (notes, assignments, grades) draft
end to end."*

This notebook is that graph, running. It is structured the same way as the Lab 6
notebook: the document's own words first, then the components, then a real run,
and finally the lab's own self-check.

Everything below is produced by execution. No number in the output is typed in
by hand, which is why the same checks cannot pass if the graph misbehaves.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))

from course_organizer.labs import lab07_node_graph as lab07

print("python  :", sys.version.split()[0])
print("package :", ROOT / "src" / "course_organizer")


python  : 3.14.7
package : C:\Users\ummea\OneDrive\Documents\Default Project\agentic-course-organizer\src\course_organizer


## The pitfall is the specification

Lab 7's stated pitfall (line 693): *"Bound the retry edge - an unbounded
regenerate loop can spin forever."*

And doc 2.12's guardrail table (line 825) gives the remedy: *"Bound MAX_ATTEMPTS
(Lab 7/10); on exhaustion, escalate to course faculty with a diagnosis."*

So the graph is not just the seven nodes. It is the seven nodes **plus a bound on
the back-edge plus an `escalate` node that carries a diagnosis**. The escalation
is the deliverable for the pitfall, not a fallback bolted on afterwards.


## 1. The document, quoted from the module that asserts it

In [2]:
from course_organizer.graph import (
    APPROVING_DECISIONS,
    DOC_BACK_EDGE,
    DOC_CHECKS,
    DOC_NODE_CHAIN,
    GATE_ROUTES,
    MAX_ATTEMPTS,
    RECURSION_BACKSTOP,
    SPECIALIST_NAMES,
    VALIDATE_ROUTES,
)
from course_organizer.graph.specialists import (
    CRITERION_WEIGHTS,
    SHINGLE_WIDTH,
    SIMILARITY_FLAG_THRESHOLD,
)

print("node chain (doc 2.10) :", " -> ".join(DOC_NODE_CHAIN))
print("back-edge             :", " ".join(DOC_BACK_EDGE))
print("validate routes       :", VALIDATE_ROUTES)
print("gate routes           :", GATE_ROUTES)
print("approving decisions   :", APPROVING_DECISIONS)
print("retry bound           :", MAX_ATTEMPTS)
print("recursion backstop    :", RECURSION_BACKSTOP, "(must never fire)")
print()
print("doc 2.7's four checks, verbatim:")
for check in DOC_CHECKS:
    print("   ", check)
print()
print("similarity flag threshold :", SIMILARITY_FLAG_THRESHOLD)
print("shingle width (words)     :", SHINGLE_WIDTH)
print("rubric criterion weights  :", CRITERION_WEIGHTS)
print("specialists               :", len(SPECIALIST_NAMES))


node chain (doc 2.10) : ingest -> plan -> workers -> validate -> assemble -> gate -> publish
back-edge             : validate workers
validate routes       : {'pass': 'assemble', 'fail': 'workers', 'escalate': 'escalate'}
gate routes           : {'approved': 'publish', 'refused': 'escalate'}
approving decisions   : ('approve', 'edit', 'override')
retry bound           : 3
recursion backstop    : 32 (must never fire)

doc 2.7's four checks, verbatim:
    draft_grade_within_rubric_maximum
    late_submissions_flagged_accurately
    similarity_score_attached
    no_grade_published_without_faculty_approval

similarity flag threshold : 0.3
shingle width (words)     : 3
rubric criterion weights  : (0.5, 0.35, 0.15)
specialists               : 6


## 2. The compiled topology, read off the graph

The seven nodes are named in the document and also declared in `state.py`. If the
wiring were wrong, those two declarations would still agree with each other. So
the topology below is read from the **compiled** graph, not from a constant.


In [3]:
import tempfile

from course_organizer.graph import (
    build_run_context,
    compiled_nodes,
    topology_lines,
)

work = Path(tempfile.mkdtemp(prefix="lab07_nb_"))
ctx = build_run_context("notebook", course_code="CS201", output_dir=work / "inspect")

print("compiled nodes:", compiled_nodes(ctx))
print()
for line in topology_lines(ctx):
    print(" ", line)


compiled nodes: ['__end__', '__start__', 'assemble', 'escalate', 'gate', 'ingest', 'plan', 'publish', 'validate', 'workers']

  compiled edges (source -> target, conditional):
    [edge] __start__ -> ingest
    [edge] assemble -> gate
    [edge] escalate -> __end__
    [cond] gate -> escalate
    [cond] gate -> publish
    [edge] ingest -> plan
    [edge] plan -> workers
    [edge] publish -> __end__
    [cond] validate -> assemble
    [cond] validate -> escalate
    [cond] validate -> workers
    [edge] workers -> validate


### The detail that makes the loop a loop

A conditional edge on its own is not a cycle. `validate --fail--> workers` only
loops because `workers --forward--> validate` is **also** present. Verified by
probe against langgraph 1.2.12: remove the forward edge and the back-edge simply
lands on `workers` once and stops.

The check below searches the compiled edge set for both directions.


In [4]:
from course_organizer.graph import graph_edges, inspect_graph

edges = graph_edges(inspect_graph(ctx))
pairs = {(src, dst) for src, dst, _conditional in edges}
conditional_pairs = {(src, dst) for src, dst, c in edges if c}

print(f"{len(edges)} compiled edge(s):")
for src, dst, conditional in edges:
    kind = "cond" if conditional else "edge"
    print(f"  [{kind}] {src} -> {dst}")
print()
print("workers -> validate present      :", ("workers", "validate") in pairs)
print("validate -> workers present      :", ("validate", "workers") in conditional_pairs)
print("escalate reaches __end__         :", ("escalate", "__end__") in pairs)


12 compiled edge(s):
  [edge] __start__ -> ingest
  [edge] assemble -> gate
  [edge] escalate -> __end__
  [cond] gate -> escalate
  [cond] gate -> publish
  [edge] ingest -> plan
  [edge] plan -> workers
  [edge] publish -> __end__
  [cond] validate -> assemble
  [cond] validate -> escalate
  [cond] validate -> workers
  [edge] workers -> validate

workers -> validate present      : True
validate -> workers present      : True
escalate reaches __end__         : True


## 3. The six Phase-A specialists live inside `workers`

Doc 2.4's roster (lines 515-523) lists the specialist agents that Lab 7 wires in.
They are not separate graph nodes in this lab: `workers` runs all six in order and
threads each one's output into the next. Lab 8 is where they fan out in parallel.


In [5]:
from course_organizer.graph import SPECIALIST_INTENTS

for position, (name, intent) in enumerate(zip(SPECIALIST_NAMES, SPECIALIST_INTENTS), 1):
    print(f"  {position}. {name}")
    print(f"      {intent}")


  1. Notes Repository Agent
      ('w1', 'Notes Repository Agent', 'read the current notes for every unit and keep their version history', 'every unit has at least one current note')
  2. Assignment Distribution Agent
      ('w2', 'Assignment Distribution Agent', 'distribute only faculty-approved assignment briefs and rubrics', 'no brief is distributed without a named approver')
  3. Submission Collection Agent
      ('w3', 'Submission Collection Agent', 'collect submissions and derive lateness without altering timestamps', 'no submission timestamp is altered')
  4. Plagiarism/Similarity Agent
      ('w4', 'Plagiarism/Similarity Agent', 'attach a similarity score and flag high cases for review, never decide', 'every graded submission carries a similarity score')
  5. Grading Assistance Agent
      ('w5', 'Grading Assistance Agent', 'draft marks against the faculty-approved rubric, leaving approval to a human', 'no draft grade exceeds the rubric maximum')
  6. Project Milestone Tracking

## 4. Doc 2.7's four validation checks

The graph's `validate` node runs exactly the four checks at doc 2.7 (line 558).
Each is a pure function of a `CheckContext`, so they can be shown directly on a
hand-built context before the real run.

First a context that should pass.


In [6]:
from course_organizer.graph import CheckContext, DraftGradeRow
from course_organizer.graph.validation import validate

def row(**overrides):
    """A grade row with sensible defaults, so each demo varies one field."""
    base = dict(
        submission_id="S001",
        student_name="A Student",
        assignment_id="A1",
        awarded_marks=15.0,
        max_marks=20.0,
        similarity_score=0.05,
        late=False,
    )
    base.update(overrides)
    return DraftGradeRow(**base)

passing = CheckContext(
    grades=[row()],
    max_marks_by_assignment={"A1": 20.0},
    late_by_submission={"S001": 3600},
)
report = validate(passing, attempt=1)
print("passed      :", report.passed)
print("checks run  :", report.checks_run)
print("failing     :", report.failing)
print("diagnosis   :", report.diagnosis or "(none)")


passed      : False
checks run  : ['draft_grade_within_rubric_maximum', 'late_submissions_flagged_accurately', 'similarity_score_attached', 'no_grade_published_without_faculty_approval']
failing     : ['late_submissions_flagged_accurately']
diagnosis   : validation failed 1 of 4 check(s): late_submissions_flagged_accurately. late_submissions_flagged_accurately: S001 is late by the store (3600s) but the grade says on time. To clear this: re-read the submission timestamps from the store; doc 2.4 forbids altering them, so the flags move, not the timestamps.


Now the same context with a grade **above** the rubric maximum. The failure has
to name the submission, the awarded mark and the maximum, or the human cannot act
on it.


In [7]:
failing = CheckContext(
    grades=[row(awarded_marks=22.0)],
    max_marks_by_assignment={"A1": 20.0},
    late_by_submission={"S001": 3600},
)
bad = validate(failing, attempt=2)
print("passed      :", bad.passed)
print("failing     :", bad.failing)
print()
print("diagnosis, as the faculty member would read it:")
print(bad.diagnosis)


passed      : False
failing     : ['draft_grade_within_rubric_maximum', 'late_submissions_flagged_accurately']

diagnosis, as the faculty member would read it:
Attempt 2 validation failed 2 of 4 check(s): draft_grade_within_rubric_maximum, late_submissions_flagged_accurately. draft_grade_within_rubric_maximum: S001 was awarded 22 against a rubric maximum of 20 for A1. late_submissions_flagged_accurately: S001 is late by the store (3600s) but the grade says on time. To clear this: re-score the affected submission(s) against the faculty-approved rubric, or raise the rubric maximum itself -- the second needs a faculty decision recorded first; re-read the submission timestamps from the store; doc 2.4 forbids altering them, so the flags move, not the timestamps.


## 5. The similarity measure is a real measurement

Doc 2.7 requires *"a plagiarism/similarity score is attached"*, and §2.12's
guardrail is blunt: *"Never make a final plagiarism/similarity determination
automatically - always flag for human review."*

So the graph **measures** and **flags**; it never adjudicates. The measure is
lexical, not invented: word-trigram shingles after stopword removal, Jaccard
overlap against every peer in the same assignment, flagging at 0.30.

An earlier version of this lab hashed the submission id to produce a score. That
was fabrication dressed as evidence, and it was removed. What follows is the real
measurement on the seed corpus, where `DS-A1-S001` and `DS-A1-S003` genuinely
differ only by `insert_after` -> `insert-after` style edits.


In [8]:
from course_organizer.graph import run_pipeline

runs_root = work / "runs"
clean = run_pipeline(
    course_code="CS201",
    run_id="clean",
    root=runs_root,
    output_dir=runs_root / "clean",
)

scores = clean.state["similarity_scores"]
against = clean.state["similarity_against"]
print(f"{len(scores)} submission(s) measured; flags at >= {SIMILARITY_FLAG_THRESHOLD}:")
print()
print(f"  {'submission':<14} {'score':>6}  {'against':<14} flagged")
for grade in clean.state["draft_grades"]:
    sid = grade.submission_id
    score = scores.get(sid)
    peer = against.get(sid) or "-"
    shown = "n/a" if score is None else f"{score:.3f}"
    print(f"  {sid:<14} {shown:>6}  {peer:<14} {grade.similarity_flagged}")


8 submission(s) measured; flags at >= 0.3:

  submission      score  against        flagged
  DS-A1-S001      1.000  DS-A1-S003     True
  DS-A1-S002      0.006  DS-A1-S001     False
  DS-A1-S003      1.000  DS-A1-S001     True
  DS-A1-S004      0.000  -              False
  DS-A2-S005      0.000  -              False
  DS-A2-S006      0.000  -              False
  DS-A3-S007      0.023  DS-A3-S008     False
  DS-A3-S008      0.023  DS-A3-S007     False


## 6. The full run, stopping at the human gate

One command: `run_pipeline` walks ingest -> plan -> workers -> validate ->
assemble and then suspends at the real `interrupt()` gate. The audit trail below
is the run's own record of what each node did.


In [9]:
print("summary :", clean.summary())
print("nodes   :", " -> ".join(clean.nodes_ran()))
print("status  :", clean.status.value)
print()
print("audit trail:")
for entry in clean.state["audit_trail"]:
    print(f"  {entry['node']:<10} {entry['summary']}")

print()
print("plan was locked before work:",
      clean.nodes_ran().index("plan") < clean.nodes_ran().index("workers"))
print("obligations locked         :", clean.state["obligation_count"])
print("drift findings             :", len(clean.state["drift_findings"]))


summary : run=clean status=draft nodes=5 attempts=1 gate=awaiting_faculty
nodes   : ingest -> plan -> workers -> validate -> assemble
status  : draft

audit trail:
  ingest     scope CS201 (Data Structures): 5 unit(s), 8 note(s), 10 version(s), 3 assignment(s); 7 scope check(s) passed
  plan       locked 16 obligation(s); plan_sha256=6d88f04886fa
  workers    pass 1: 6 specialist(s), 0 reported a problem
  validate   pass 1: all 4 check(s) passed
  assemble   assembled DOC-CS201-organised-course-4f12bcb7d9 (8 section(s)); drift findings: 0

plan was locked before work: True
obligations locked         : 16
drift findings             : 0


### What the human is asked

Doc 2.11 describes the dashboard as *"Rubric-aligned draft grades and feedback
awaiting faculty approval"*. The gate request is exactly that: the grades, the
similarity flags, and the validation report that cleared them - and every grade
still marked `draft`.


In [10]:
request = clean.gate_request
print("course      :", request.course_code, request.course_name)
print("attempt     :", request.attempt)
print("units       :", ", ".join(request.units_covered))
print("assignments :", ", ".join(request.assignments_included))
print("validation  :", "passed" if request.validation.passed else "FAILED")
print("grades      :", len(request.draft_grades))
print("all draft   :", {g.status for g in request.draft_grades} == {"draft"})
print("approved by :", {g.approved_by for g in request.draft_grades})
print()
print(f"  {'submission':<14} {'student':<16} {'awarded':>8} {'max':>6}  late  sim  flag")
for grade in request.draft_grades:
    sim = "n/a" if grade.similarity_score is None else f"{grade.similarity_score:.2f}"
    print(
        f"  {grade.submission_id:<14} {grade.student_name:<16} "
        f"{grade.awarded_marks:>8.2f} {grade.max_marks:>6.1f}  "
        f"{'yes' if grade.late else 'no':<4} {sim:>5}  "
        f"{'*' if grade.similarity_flagged else ''}"
    )
print()
print("flagged for review (not adjudicated):")
for flag in request.similarity_flags:
    print(f"  {flag['submission_id']} vs {flag['against']} = {flag['score']}")


course      : CS201 Data Structures
attempt     : 1
units       : 1, 2, 3, 4, 5
assignments : ASG-01, ASG-02, ASG-03
validation  : passed
grades      : 8
all draft   : True
approved by : {''}

  submission     student           awarded    max  late  sim  flag
  DS-A1-S001     Aarav Menon         15.89   20.0  no    1.00  *
  DS-A1-S002     Diya Kulkarni        9.33   20.0  no    0.01  
  DS-A1-S003     Kabir Shah          18.88   20.0  no    1.00  *
  DS-A1-S004     Meera Nair          11.77   20.0  no    0.00  
  DS-A2-S005     Rohan Pillai         8.23   20.0  no    0.00  
  DS-A2-S006     Sara Thomas         15.56   20.0  no    0.00  
  DS-A3-S007     Vivaan Rao          11.01   25.0  no    0.02  
  DS-A3-S008     Ananya Bose         17.13   25.0  yes   0.02  

flagged for review (not adjudicated):
  DS-A1-S001 vs DS-A1-S003 = 1.0
  DS-A1-S003 vs DS-A1-S001 = 1.0


## 7. The back-edge, and the bound

`fail_until_attempt` injects a defect for the first *n* passes, so the sequence of
real executions shows whether the back-edge fired. This is not a simulation of a
retry: the compiled graph runs `workers` and `validate` again each time.

Then `fail_until_attempt=99` never converges, and the bound is what stops it.


In [11]:
retry = run_pipeline(
    course_code="CS201",
    run_id="retry",
    root=runs_root,
    output_dir=runs_root / "retry",
    fail_until_attempt=2,
)
print("summary      :", retry.summary())
print("nodes        :", " -> ".join(retry.nodes_ran()))
print("workers ran  :", retry.nodes_ran().count("workers"), "time(s)")
print()
print("attempt log:")
for entry in retry.state["attempt_log"]:
    faulty = [s["step_id"] for s in entry["specialists"] if not s["ok"]]
    print(f"  pass {entry['pass']}:", f"faulty {faulty}" if faulty else "all specialists ok")


summary      : run=retry status=draft nodes=9 attempts=3 gate=awaiting_faculty
nodes        : ingest -> plan -> workers -> validate -> workers -> validate -> workers -> validate -> assemble
workers ran  : 3 time(s)

attempt log:
  pass 1: faulty ['w4', 'w5']
  pass 2: faulty ['w4', 'w5']
  pass 3: all specialists ok


In [12]:
exhausted = run_pipeline(
    course_code="CS201",
    run_id="exhaustion",
    root=runs_root,
    output_dir=runs_root / "exhaustion",
    fail_until_attempt=99,
)
esc = exhausted.state["escalation"]
print("summary    :", exhausted.summary())
print("nodes      :", " -> ".join(exhausted.nodes_ran()))
print("attempts   :", exhausted.attempts_made(), "of", MAX_ATTEMPTS)
print("reason     :", esc.reason)
print("raised by  :", esc.raised_by, "->", esc.escalated_to)
print("failing    :", ", ".join(esc.failing))
print("draft kept :", len(esc.pending_draft.get("draft_grades", [])), "grade(s)")
print()
print("diagnosis:")
print(esc.diagnosis)


escalate escalated to course_faculty: retry_budget_exhausted after 3 attempt(s)


summary    : run=exhaustion status=escalated nodes=9 attempts=3 escalated=retry_budget_exhausted
nodes      : ingest -> plan -> workers -> validate -> workers -> validate -> workers -> validate -> escalate
attempts   : 3 of 3
reason     : retry_budget_exhausted
raised by  : validate -> course_faculty
failing    : draft_grade_within_rubric_maximum, similarity_score_attached
draft kept : 8 grade(s)

diagnosis:
Validation still failing after 3 of 3 permitted attempt(s); the run stopped rather than retrying forever. Attempt 3 validation failed 2 of 4 check(s): draft_grade_within_rubric_maximum, similarity_score_attached. draft_grade_within_rubric_maximum: DS-A1-S001 was awarded 22 against a rubric maximum of 20 for ASG-01; DS-A1-S002 was awarded 22 against a rubric maximum of 20 for ASG-01; DS-A1-S003 was awarded 22 against a rubric maximum of 20 for ASG-01; DS-A1-S004 was awarded 22 against a rubric maximum of 20 for ASG-01; DS-A2-S005 was awarded 22 against a rubric maximum of 20 for ASG

### The bound is not the recursion limit

`recursion_limit` (here 32) is a backstop against a miswired graph; it must never
be what stops a legitimate retry, because it raises `GraphRecursionError` rather
than producing a diagnosis. The bound that *does* the work is the explicit
`attempts` counter in the state, which routes to `escalate`. Both are asserted:
the retry above converged in three passes, and the exhaustion run stopped at
exactly `MAX_ATTEMPTS`.


## 8. A human says no

The gate is a hard stop, and a refusal is as much an outcome as an approval. A
refusal publishes nothing and leaves every grade unapproved.


In [13]:
from course_organizer.graph import DECISION_REFUSE, resume_run

refusal = run_pipeline(
    course_code="CS201",
    run_id="refusal",
    root=runs_root,
    output_dir=runs_root / "refusal",
)
refused = resume_run("refusal", DECISION_REFUSE, root=runs_root)

print("summary   :", refused.summary())
print("status    :", refused.status.value)
print("published :", refused.published)
print("path      :", refused.state.get("published_path"))
print("escalation:", refused.state["escalation"].reason)
print("all draft :", {g.status for g in refused.state["draft_grades"]} == {"draft"})


escalate escalated to course_faculty: refuse after 1 attempt(s)


summary   : run=refusal status=refused nodes=7 attempts=1 escalated=refuse
status    : refused
published : False
path      : None
escalation: refuse
all draft : True


## 9. A suspended run survives the process

The gate request is written into the SQLite checkpointer, so a *different*
interpreter can read it back and resume. This is the difference between a pause
and a crash: `resume_run` reads the request from disk, and the nodes before the
gate do not re-run, because the state they produced is already checkpointed.


In [14]:
from course_organizer.graph import FacultyDecision, DECISION_APPROVE, load_gate_request

recovered = load_gate_request("clean", root=runs_root)
print("recovered from SQLite :", recovered is not None)
print("grades in the request :", len(recovered.draft_grades))
print("row type              :", type(recovered.draft_grades[0]).__name__)
print()

approved = resume_run(
    "clean",
    FacultyDecision(
        decision=DECISION_APPROVE,
        decided_by="Dr. Iyer",
        reason="marks and flags both check out",
    ),
    root=runs_root,
)

print("summary after resume  :", approved.summary())
print("nodes after resume    :", " -> ".join(approved.nodes_ran()))
print("pre-gate nodes re-ran :", approved.nodes_ran().count("ingest"), "time(s)")
print("published             :", approved.published)

published_path = Path(approved.state["published_path"])
body = published_path.read_text(encoding="utf-8")
print("published file        :", published_path.name, f"({len(body)} chars)")
print("says it is published  :", "Published: yes" in body)
print("names the approver    :", "Dr. Iyer" in body)
print("pure ASCII            :", all(ord(ch) < 128 for ch in body))


recovered from SQLite : True
grades in the request : 8
row type              : DraftGradeRow

summary after resume  : run=clean status=published nodes=7 attempts=1 published=C:\Users\ummea\AppData\Local\Temp\lab07_nb_0nq1di7z\runs\clean\DOC-CS201-organised-course-4f12bcb7d9.md
nodes after resume    : ingest -> plan -> workers -> validate -> assemble -> gate -> publish
pre-gate nodes re-ran : 1 time(s)
published             : True
published file        : DOC-CS201-organised-course-4f12bcb7d9.md (4695 chars)
says it is published  : True
names the approver    : True
pure ASCII            : True


The last two lines matter. Lab 3's formatter is hard-wired for a draft and emits
`Approved : no` and `Published: no`; a published artifact carrying those lines
would contradict itself above a table of published grades. The `publish` node
stamps the pre-gate lines rather than editing Lab 3's tested module.


## 10. The whole lab, self-checking

`run_scenarios` runs all four scenarios in a fresh directory and asserts the
acceptance checks against what actually happened.


In [15]:
result = lab07.run_scenarios(work / "full")

print("pass_all :", result.pass_all)
print()
for number, check in enumerate(result.checks, 1):
    mark = "PASS" if check["passed"] else "FAIL"
    print(f"[{mark}] {number:>2}. {check['name']}")
    print(f"        {check['detail']}")
print()
print("=" * 72)
print(f"OVERALL: {'PASS' if result.pass_all else 'FAIL'} "
      f"({sum(1 for c in result.checks if c['passed'])}/{len(result.checks)} checks)")


escalate escalated to course_faculty: retry_budget_exhausted after 3 attempt(s)


escalate escalated to course_faculty: refuse after 1 attempt(s)


pass_all : True

[PASS]  1. every doc node is compiled into the graph
        doc chain ingest, plan, workers, validate, assemble, gate, publish plus escalate
[PASS]  2. the back-edge exists: validate --fail--> workers
        compiled edges include validate -> workers (conditional)
[PASS]  3. workers --forward--> validate, without which the loop cannot cycle
        a conditional edge alone does not create a cycle
[PASS]  4. the clean run executes the doc path and stops at the gate
        nodes executed: ingest -> plan -> workers -> validate -> assemble
[PASS]  5. the clean run stops at a real human gate
        run=clean status=draft nodes=5 attempts=1 gate=awaiting_faculty
[PASS]  6. all four of doc 2.7's checks ran and passed
        4 check(s): draft_grade_within_rubric_maximum, late_submissions_flagged_accurately, similarity_score_attached, no_grade_published_without_faculty_approval
[PASS]  7. nothing published before a human approved
        status=draft, no published file
[PA

### The same lab, as a command

The module is also a CLI. Running it is what the evidence file
`docs/evidence/lab07_cli.txt` captures; here it is run from inside the notebook so
the exit code is visible too.


In [16]:
import os
import subprocess

proc = subprocess.run(
    [sys.executable, "-m", "course_organizer.labs.lab07_node_graph"],
    capture_output=True,
    text=True,
    timeout=600,
    env={**os.environ, "PYTHONPATH": str(ROOT / "src")},
)
final = [ln for ln in proc.stdout.splitlines() if ln.strip()]
print("last stdout line :", final[-1])
print("exit code        :", proc.returncode)
assert proc.returncode == 0, proc.stdout + proc.stderr


last stdout line : ==============================================================================
exit code        : 0


## 11. Requirement coverage

| Doc reference | Requirement | Where it is met |
| --- | --- | --- |
| 2.10 L690 | One command yields a validated workspace draft end to end | `run_pipeline`, section 6 |
| 2.10 L696 | `ingest -> plan -> workers -> validate`, `validate --fail--> workers`, `--pass--> assemble -> gate -> publish` | compiled topology, section 2 |
| 2.10 L692 | All core nodes plus the back-edge | `compiled_nodes`, section 2 |
| 2.10 L693 | Bound the retry edge | explicit `attempts`, section 7 |
| 2.7 L558 | A draft grade never exceeds the rubric maximum | `draft_grade_within_rubric_maximum`, section 4 |
| 2.7 L558 | Late submissions flagged accurately | `late_submissions_flagged_accurately` |
| 2.7 L558 | A plagiarism/similarity score is attached | `similarity_score_attached`, section 5 |
| 2.7 L558 | No grade published without faculty approval | `no_grade_published_without_faculty_approval`, sections 6 and 8 |
| 2.12 L825 | On exhaustion, escalate with a diagnosis | `escalate` node, section 7 |
| 2.12 L830 | Never publish without faculty approval | `publish` re-checks, section 8 |
| 2.4 L515-523 | The specialist agents | the six inside `workers`, section 3 |

The coverage table says *where*; the acceptance checks in section 10 say whether.


## 12. Reproducing this

```
python -m pytest tests/test_lab07_graph.py tests/test_lab07_runner.py
python -m course_organizer.labs.lab07_node_graph
python tools/nb_to_text.py notebooks/lab07_node_graph.ipynb docs/evidence/lab07_notebook_output.txt
```

No API key and no network: the default `LLM_BACKEND=scripted` runs the whole lab,
which is what makes the acceptance criteria re-runnable by a reviewer.
